# Week 6 (Stretch) — Do Deep-Network Features Predict the Brain Better?

Our baseline used a simple summary of the image (shrunk pixels + PCA). Modern vision AI uses **deep neural networks (CNNs)** trained on millions of images. A natural question:

> Do features from a pretrained CNN predict V1 activity better than raw pixels? And does behavior still help on top of them?

We use a pretrained **ResNet-50** purely as a *feature extractor* (we don't train it), take a **middle layer** (middle layers tend to resemble early visual cortex), and feed those features into the same ridge-regression setup as before.

> **Turn on the GPU first:** *Runtime → Change runtime type → T4 GPU*. This notebook needs it for the feature extraction step.

## 1. Get the data (same mouse as before)

In [ ]:
FILENAME = "static21067-10-18-GrayImageNet-94c6ff995dac583098847cfecd43e7b6"
URL = f"https://gin.g-node.org/cajal/Sensorium2022/raw/master/{FILENAME}.zip"
import os
if not os.path.exists(FILENAME):
    print("Downloading (~424 MB)...")
    !wget -q --show-progress -O mouse.zip "$URL"
    !unzip -q -o mouse.zip
print("✅ Data ready.")

In [ ]:
import numpy as np, torch
print("GPU available:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only (turn on the GPU!)")
device = "cuda" if torch.cuda.is_available() else "cpu"

DATA_DIR = f"{FILENAME}/data"
n = len(os.listdir(f"{DATA_DIR}/responses")); n_neurons = np.load(f"{DATA_DIR}/responses/0.npy").shape[0]
imgs = np.zeros((n,144,256), np.float32); R = np.zeros((n,n_neurons), np.float32); B = np.zeros((n,3))
for i in range(n):
    imgs[i]=np.load(f"{DATA_DIR}/images/{i}.npy")[0]; R[i]=np.load(f"{DATA_DIR}/responses/{i}.npy"); B[i]=np.load(f"{DATA_DIR}/behavior/{i}.npy").ravel()
tiers = np.load(f"{FILENAME}/meta/trials/tiers.npy", allow_pickle=True)
is_tr = tiers=="train"; is_va = tiers=="validation"; is_te = tiers=="test"
print("loaded", imgs.shape)

## 2. Extract features from a pretrained CNN

We take ResNet-50's layer-2 output at the image's native resolution (so the spatial map — *where* things are — is preserved, which matters a lot for visual cortex), then pool it to a 9×16 grid. You can treat this cell as a feature-extraction tool.

In [ ]:
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights

# luminance-normalize images to [0,1]
lo, hi = np.percentile(imgs[::30],1), np.percentile(imgs[::30],99)
imgs_n = np.clip((imgs-lo)/(hi-lo+1e-6), 0, 1)

net = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2).to(device).eval()
extractor = nn.Sequential(net.conv1, net.bn1, net.relu, net.maxpool, net.layer1, net.layer2).to(device).eval()
pool = nn.AdaptiveAvgPool2d((9,16))
mean = torch.tensor([0.485,0.456,0.406],device=device).view(1,3,1,1)
std  = torch.tensor([0.229,0.224,0.225],device=device).view(1,3,1,1)

def extract(batch):
    x = torch.from_numpy(batch).to(device).unsqueeze(1).repeat(1,3,1,1)  # grayscale -> 3 channels
    x = (x-mean)/std
    with torch.no_grad(): f = pool(extractor(x))
    return f.flatten(1).cpu().numpy()

CF = np.zeros((n, 512*9*16), np.float32); bs = 64
for s in range(0, n, bs):
    CF[s:s+bs] = extract(imgs_n[s:s+bs])
print("CNN features:", CF.shape)

## 3. Compare four models fairly

We compare pixel vs CNN features, each with and without behavior. To be fair, we pick each model's regularization (`alpha`) using the **validation** trials, and report the final score on the **test** trials.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

# pixel features (same as Week 3)
px = imgs_n.reshape(n,36,4,64,4).mean(axis=(2,4)).reshape(n,-1)
PX = PCA(150, random_state=0).fit(StandardScaler().fit_transform(px)[is_tr]).transform(StandardScaler().fit_transform(px))
# cnn features -> PCA 800
CFs = StandardScaler().fit(CF[is_tr]).transform(CF)
CFp = PCA(800, svd_solver="randomized", random_state=0).fit(CFs[is_tr]).transform(CFs)
Bs  = StandardScaler().fit(B[is_tr]).transform(B)

def corr(P,A): P=P-P.mean(0); A=A-A.mean(0); return (P*A).sum(0)/(np.sqrt((P**2).sum(0)*(A**2).sum(0))+1e-12)
def evaluate(X):
    best=None
    for a in [1e2,1e3,1e4,1e5]:
        m=Ridge(alpha=a).fit(X[is_tr],R[is_tr]); v=np.nanmean(corr(m.predict(X[is_va]),R[is_va]))
        if best is None or v>best[1]: best=(m,v)
    return np.nanmean(corr(best[0].predict(X[is_te]),R[is_te]))

results = {
    "pixel only":      evaluate(PX),
    "pixel + behavior": evaluate(np.hstack([PX,Bs])),
    "CNN only":         evaluate(CFp),
    "CNN + behavior":   evaluate(np.hstack([CFp,Bs])),
}
for k,v in results.items(): print(f"{k:18s}: {v:.3f}")

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(6.5,4))
bars = plt.bar(list(results.keys()), list(results.values()),
               color=["#bbb","#9bbf9b","#7d9fc9","#3a7d44"])
for b,v in zip(bars, results.values()): plt.text(b.get_x()+b.get_width()/2, v+0.003, f"{v:.3f}", ha="center")
plt.ylabel("mean test correlation"); plt.xticks(rotation=15)
plt.title("Deep-network features + behavior predict V1 best")
plt.tight_layout(); plt.show()

## 4. What we learned

- **CNN features beat raw pixels** for the stimulus model (about 0.137 vs 0.131) — middle-layer features of a network trained on natural images are a little more "V1-like" than raw pixels. The gain is real but modest, which makes sense: an ImageNet network wasn't trained to match a mouse brain.
- **Behavior still helps on top of the CNN** (about 0.137 → 0.155). The best model is **CNN + behavior**.
- Most importantly: **the behavior effect is robust.** It improves prediction whether the stimulus model is simple (pixels) or fancy (CNN). Our main conclusion doesn't depend on the modeling choice.

> **A good scientist's takeaway:** a fancier stimulus model gave a small boost, but the internal-state (arousal) signal remained about as important as switching model types. Knowing *what the animal is doing* is not a minor add-on — it's a first-class predictor of neural activity.

You've now compared a classic method and a deep-learning method, fairly, on real data — and drawn an honest conclusion. That's exactly how research works. 🧠🤖